# LlamaIndex: consultas em linguagem natural sobre DataFrames
Uso do `PandasQueryEngine` do LlamaIndex com uma LLM da Groq para responder perguntas sobre um DataFrame,
customização de prompts e saídas com um pipeline (Workflow), geração de relatório em PDF e uma aplicação
Gradio que analisa qualquer arquivo CSV.

**Origem:** Alura — curso de LlamaIndex com Pandas (consultas com `PandasQueryEngine`), com adaptações próprias.

**Dados:** `vendas.csv` lido direto do GitHub (https://raw.githubusercontent.com/alura-cursos/llamaIndex_pandas_query/refs/heads/main/Dados/vendas.csv).
Na aplicação Gradio (seção 6) qualquer CSV pode ser enviado.

**Secrets do Colab:** `GROQ_API` (chave da API da Groq).

**Índice**
1. Setup: dados e LLM
2. Consultas com PandasQueryEngine
3. Customizando as saídas
4. Customizando os prompts: pipeline com Workflow
5. Relatório em PDF
6. Aplicação Gradio para qualquer CSV

In [ ]:
!pip install -q -U llama-index llama-index-experimental llama-index-llms-groq gradio fpdf2 chardet

import textwrap
from datetime import datetime

import chardet
import gradio as gr
import pandas as pd
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from google.colab import userdata
from llama_index.core import PromptTemplate, Settings
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.experimental.query_engine import PandasQueryEngine
from llama_index.experimental.query_engine.pandas import PandasInstructionParser
from llama_index.llms.groq import Groq

## 1. Setup: dados e LLM
Carregamos a base de vendas e configuramos a LLM da Groq como modelo padrão do LlamaIndex (`Settings.llm`).
A chave da API fica nos *secrets* do Colab e é lida com `userdata.get`.

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/llamaIndex_pandas_query/refs/heads/main/Dados/vendas.csv'
df = pd.read_csv(url)
df.head()

In [ ]:
df.info()
print('Linhas duplicadas:', df.duplicated().sum())

In [ ]:
key = userdata.get('GROQ_API')

# Modelo usado em todo o notebook (o curso usava 'llama3-70b-8192', descontinuado na Groq)
llm = Groq(model='openai/gpt-oss-20b', api_key=key)
Settings.llm = llm

## 2. Consultas com PandasQueryEngine
O `PandasQueryEngine` converte a pergunta em linguagem natural em código Pandas, executa o código sobre o
DataFrame e devolve o resultado. Com `verbose=True` vemos as instruções Pandas geradas pela LLM.
Depois de cada consulta conferimos o resultado com o código Pandas equivalente.

In [ ]:
query_engine = PandasQueryEngine(df=df, verbose=True)

In [ ]:
response = query_engine.query('Qual é a forma de pagamento mais utilizada pelos clientes?')

In [ ]:
df['forma_pagamento'].value_counts()

In [ ]:
response = query_engine.query('Qual é o tipo de produto com maior quantidade por filial?')
print(response.response)

In [ ]:
df.groupby(['filial', 'tipo_produto'])['quantidade'].sum()

## 3. Customizando as saídas
### 3.1 Respostas com explicações
Com `synthesize_response=True` a LLM recebe a saída do Pandas e sintetiza uma resposta explicativa,
em vez de devolver apenas o resultado bruto.

In [ ]:
query_engine = PandasQueryEngine(df=df, verbose=True, synthesize_response=True)

response = query_engine.query('Qual é a avaliação média de cada filial?')
print(str(response))

In [ ]:
df.groupby('filial')['avaliacao'].mean()

In [ ]:
def formatar_texto(response):
    texto = response.response
    texto_formatado = textwrap.fill(texto, width=100)
    print(texto_formatado)

formatar_texto(response)

### 3.2 Obtendo gráficos
A forma de pedir importa: "exibir" tende a gerar uma descrição textual, enquanto "plotar" faz a LLM gerar
código de visualização (matplotlib) que é executado sobre o DataFrame.

In [ ]:
response = query_engine.query('Você pode plotar a distribuição das avaliações?')
formatar_texto(response)

In [ ]:
response = query_engine.query('''Plote o valor total por tipo de produto,
ordenando do menor para o maior e mantendo as barras na horizontal''')
formatar_texto(response)

### 3.3 Direcionando o idioma das respostas
Sem instrução explícita a resposta sintetizada pode vir em inglês; basta pedir o idioma na pergunta.

In [ ]:
response = query_engine.query('Qual é o método de pagamento mais utilizado em cada filial?')
print(response.response)
print('-' * 30)
response = query_engine.query('Qual é o método de pagamento mais utilizado em cada filial? Responda em português BR')
print(response.response)

In [ ]:
df.groupby('filial')['forma_pagamento'].value_counts()

## 4. Customizando os prompts: pipeline com Workflow
Para controlar cada etapa, montamos o pipeline manualmente com três módulos:
- **pandas_prompt**: traduz a pergunta em linguagem natural para uma expressão Pandas. Recebe as instruções,
  o tipo de cada coluna e `df.head()` para que o modelo entenda o contexto dos dados;
- **pandas_output_parser** (`PandasInstructionParser`): executa a expressão gerada sobre o DataFrame
  (internamente usa `eval()`, que executa expressões Python armazenadas em strings);
- **response_synthesis_prompt**: com a pergunta, o código e a saída do Pandas, a LLM atua como analista de dados,
  responde de forma natural e exibe ao final o código utilizado.

No curso o encadeamento era feito com `QueryPipeline` (`add_chain`/`add_links`), removido nas versões recentes
do LlamaIndex. Aqui usamos a API de **Workflows**: cada método decorado com `@step` é uma etapa, e as ligações
são inferidas pelos tipos de evento de entrada e saída (`StartEvent` → `PandasEvent` → `StopEvent`).

In [ ]:
# Função para obter uma descrição das colunas do DataFrame
def descricao_colunas(df):
    descricao = '\n'.join([f"`{col}`: {str(df[col].dtype)}" for col in df.columns])
    return 'Aqui estão os detalhes das colunas do DataFrame:\n' + descricao

# Instruções para orientar o modelo a converter a consulta em código Python executável com Pandas
instruction_str = (
    "1. Converta a consulta para código Python executável usando Pandas.\n"
    "2. A linha final do código deve ser uma expressão Python que possa ser chamada com a função `eval()`.\n"
    "3. O código deve representar uma solução para a consulta.\n"
    "4. IMPRIMA APENAS A EXPRESSÃO.\n"
    "5. Não coloque a expressão entre aspas.\n")

# Prompt enviado ao modelo para gerar o código Pandas
pandas_prompt_str = (
    "Você está trabalhando com um dataframe do pandas em Python chamado `df`.\n"
    "{colunas_detalhes}\n\n"
    "Este é o resultado de `print(df.head())`:\n"
    "{df_str}\n\n"
    "Siga estas instruções:\n"
    "{instruction_str}\n"
    "Consulta: {query_str}\n\n"
    "Expressão:"
)

# Prompt para sintetizar a resposta a partir dos resultados da consulta Pandas
response_synthesis_prompt_str = (
    "Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.\n"
    "Responda de forma natural, sem introduções como 'A resposta é:' ou algo semelhante.\n"
    "Consulta: {query_str}\n\n"
    "Instruções do Pandas (opcional):\n{pandas_instructions}\n\n"
    "Saída do Pandas: {pandas_output}\n\n"
    "Resposta: \n\n"
    "Ao final, exibir o código usado para gerar a resposta, no formato: O código utilizado foi `{pandas_instructions}`"
)
response_synthesis_prompt = PromptTemplate(response_synthesis_prompt_str)

In [ ]:
# Evento que carrega os dados entre o 1º e o 2º step (cada atributo é um campo obrigatório)
class PandasEvent(Event):
    query_str: str            # pergunta original do usuário
    pandas_instructions: str  # código Pandas gerado pela LLM
    pandas_output: str        # resultado da execução do código sobre o df


class PandasWorkflow(Workflow):
    """Pipeline pergunta -> código Pandas -> execução -> resposta sintetizada, para um DataFrame qualquer."""

    def __init__(self, df, llm, **kwargs):
        super().__init__(**kwargs)
        self.llm = llm
        # Módulo que traduz a pergunta em instruções Pandas
        self.pandas_prompt = PromptTemplate(pandas_prompt_str).partial_format(
            instruction_str=instruction_str, colunas_detalhes=descricao_colunas(df), df_str=df.head(5)
        )
        # Módulo que executa as instruções Pandas sobre o DataFrame
        self.pandas_output_parser = PandasInstructionParser(df)

    # 1º step: recebe o StartEvent (dados passados em w.run(...)) e devolve um PandasEvent
    @step
    async def gerar_instrucoes_pandas(self, ev: StartEvent) -> PandasEvent:
        query_str = ev.query_str
        prompt_formatado = self.pandas_prompt.format(query_str=query_str)
        resposta_llm1 = await self.llm.acomplete(prompt_formatado)   # equivalente ao antigo módulo "llm1"
        pandas_instructions = str(resposta_llm1)
        pandas_output = self.pandas_output_parser.parse(pandas_instructions)
        return PandasEvent(
            query_str=query_str,
            pandas_instructions=pandas_instructions,
            pandas_output=str(pandas_output),
        )

    # 2º step: recebe o PandasEvent e devolve o StopEvent, que encerra o workflow com o resultado final
    @step
    async def sintetizar_resposta(self, ev: PandasEvent) -> StopEvent:
        prompt_formatado = response_synthesis_prompt.format(
            query_str=ev.query_str,
            pandas_instructions=ev.pandas_instructions,
            pandas_output=ev.pandas_output,
        )
        resposta_llm2 = await self.llm.acomplete(prompt_formatado)   # equivalente ao antigo módulo "llm2"
        return StopEvent(result=str(resposta_llm2))


def pipeline_consulta(df):
    # timeout: tempo máximo (s) de execução; verbose=True mostra qual step está rodando
    return PandasWorkflow(df, llm, timeout=120, verbose=False)

In [ ]:
w = pipeline_consulta(df)

pergunta_1 = 'Qual é a média gasta por cada tipo de cliente?'
resposta_1 = await w.run(query_str=pergunta_1)
print(textwrap.fill(resposta_1, width=100))

In [ ]:
pergunta_2 = 'Por que clientes tipo membro tem maior média de gasto?'
resposta_2 = await w.run(query_str=pergunta_2)
print(textwrap.fill(resposta_2, width=100))

## 5. Relatório em PDF
O histórico de perguntas e respostas é guardado como uma lista de tuplas e exportado para PDF com a `fpdf2`.
As fontes padrão do PDF só aceitam caracteres latin-1, por isso caracteres fora dessa tabela (emojis, aspas
tipográficas etc. que a LLM pode gerar) são substituídos antes de escrever.

In [ ]:
# Função para adicionar a pergunta e a resposta ao histórico
def add_historico(pergunta, resposta, historico_estado):
    if pergunta and resposta:
        historico_estado.append((pergunta, resposta))
        gr.Info('Adicionado ao PDF!', duration=2)
    return historico_estado


def texto_pdf(texto):
    return str(texto).encode('latin-1', 'replace').decode('latin-1')


# Função para gerar o PDF
def gerar_pdf(historico_estado):
    if not historico_estado:
        gr.Warning('Nenhum dado para adicionar ao PDF.')
        return None

    # Nome do arquivo com timestamp
    timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
    caminho_pdf = f'relatorio_perguntas_respostas_{timestamp}.pdf'

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    for pergunta, resposta in historico_estado:
        pdf.set_font('Helvetica', 'B', 14)
        pdf.multi_cell(0, 8, texto_pdf(pergunta), new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(2)
        pdf.set_font('Helvetica', '', 12)
        pdf.multi_cell(0, 8, texto_pdf(resposta), new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(6)

    pdf.output(caminho_pdf)
    return caminho_pdf

In [ ]:
# Relatório com as duas consultas da seção 4
historico = [(pergunta_1, resposta_1), (pergunta_2, resposta_2)]
gerar_pdf(historico)

## 6. Aplicação Gradio para qualquer CSV
A interface permite enviar um CSV, ver uma prévia dos dados, fazer perguntas (respondidas pelo `PandasWorkflow`),
adicionar as interações ao histórico, gerar o PDF e recomeçar com outro dataset.
Para aceitar arquivos variados, a codificação é detectada com `chardet` e o separador é inferido pelo Pandas
(`sep=None, engine='python'`).

In [ ]:
# Leitura de um CSV qualquer: detecta a codificação e o separador
def ler_csv(caminho_arquivo):
    with open(caminho_arquivo, 'rb') as f:
        deteccao = chardet.detect(f.read(100_000))
    encoding = deteccao['encoding'] or 'utf-8'
    return pd.read_csv(caminho_arquivo, sep=None, engine='python', encoding=encoding)


# Função para carregar os dados
def carregar_dados(caminho_arquivo, df_estado):
    if caminho_arquivo is None or caminho_arquivo == '':
        return 'Por favor, faça o upload de um arquivo CSV para analisar.', pd.DataFrame(), df_estado
    try:
        df = ler_csv(caminho_arquivo)
        return 'Arquivo carregado com sucesso!', df.head(), df
    except Exception as e:
        return f'Erro ao carregar arquivo: {str(e)}', pd.DataFrame(), df_estado


# Função para processar a pergunta
async def processar_pergunta(pergunta, df_estado):
    if df_estado is not None and pergunta:
        w = pipeline_consulta(df_estado)
        return await w.run(query_str=pergunta)
    return ''


# Função para limpar a pergunta e a resposta
def limpar_pergunta_resposta():
    return '', ''


# Função para resetar a aplicação
def resetar_aplicacao():
    return None, 'A aplicação foi resetada. Por favor, faça upload de um novo arquivo CSV.', pd.DataFrame(), '', None, [], ''

In [ ]:
with gr.Blocks() as app:

    gr.Markdown('# Analisando os dados🔎🎲')
    gr.Markdown("""
    Carregue um arquivo CSV e faça perguntas sobre os dados. A cada pergunta, você poderá
    visualizar a resposta e, se desejar, adicionar essa interação ao PDF final, basta clicar
    em "Adicionar ao histórico do PDF". Para fazer uma nova pergunta, clique em "Limpar pergunta e resultado".
    Após definir as perguntas e respostas no histórico, clique em "Gerar PDF". Assim, será possível
    baixar um PDF com o registro completo das suas interações. Se você quiser analisar um novo dataset,
    basta clicar em "Quero analisar outro dataset" ao final da página.
    """)

    input_arquivo = gr.File(file_count='single', type='filepath', label='Upload CSV')
    upload_status = gr.Textbox(label='Status do Upload:')
    tabela_dados = gr.DataFrame()

    gr.Markdown("""
    Exemplos de perguntas:
    1. Qual é o número de registros no arquivo?
    2. Quais são os tipos de dados das colunas?
    3. Quais são as estatísticas descritivas das colunas numéricas?
    """)

    input_pergunta = gr.Textbox(label='Digite sua pergunta sobre os dados')
    botao_submeter = gr.Button('Enviar')
    output_resposta = gr.Textbox(label='Resposta')

    with gr.Row():
        botao_limpeza = gr.Button('Limpar pergunta e resultado')
        botao_add_pdf = gr.Button('Adicionar ao histórico do PDF')
        botao_gerar_pdf = gr.Button('Gerar PDF')

    arquivo_pdf = gr.File(label='Download do PDF')
    botao_resetar = gr.Button('Quero analisar outro dataset!')

    # Estados da sessão
    df_estado = gr.State(value=None)       # DataFrame carregado
    historico_estado = gr.State(value=[])  # histórico de perguntas e respostas

    # Conectando funções aos componentes
    input_arquivo.change(fn=carregar_dados, inputs=[input_arquivo, df_estado],
                         outputs=[upload_status, tabela_dados, df_estado])
    botao_submeter.click(fn=processar_pergunta, inputs=[input_pergunta, df_estado], outputs=output_resposta)
    botao_limpeza.click(fn=limpar_pergunta_resposta, inputs=[], outputs=[input_pergunta, output_resposta])
    botao_add_pdf.click(fn=add_historico, inputs=[input_pergunta, output_resposta, historico_estado],
                        outputs=historico_estado)
    botao_gerar_pdf.click(fn=gerar_pdf, inputs=[historico_estado], outputs=arquivo_pdf)
    botao_resetar.click(fn=resetar_aplicacao, inputs=[],
                        outputs=[input_arquivo, upload_status, tabela_dados, output_resposta,
                                 arquivo_pdf, historico_estado, input_pergunta])

app.launch(debug=True, share=True, theme='soft')

### 6.1 Publicando no Hugging Face Spaces
Para publicar, crie um Space Gradio com um `app.py` contendo as funções das seções 4 a 6 (sem `userdata`) e
leia a chave com `os.getenv('GROQ_API')`, cadastrando-a como *secret* do Space. Ao final do arquivo use
`if __name__ == "__main__": app.launch()`. O `requirements.txt` do Space deve listar:
`pandas`, `gradio`, `llama-index`, `llama-index-llms-groq`, `llama-index-experimental`, `fpdf2` e `chardet`.